# Managing short-term memory: trimming

The checkpointer still saves the **full** history, but before each LLM call we **trim** it so the
model only sees the most recent messages that fit in a token budget. The prompt stops growing.

Trade-off: whatever was trimmed off, the bot can no longer see.

In [1]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, trim_messages
from langchain_core.messages.utils import count_tokens_approximately
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

model = ChatGroq(model='openai/gpt-oss-20b', reasoning_effort='low')

In [2]:
# First, trim_messages on its own: keep the LAST messages that fit in max_tokens
history = [
    SystemMessage('You are a helpful assistant.'),
    HumanMessage('My name is Priyansh.'),
    AIMessage('Nice to meet you, Priyansh!'),
    HumanMessage('I live in Jaipur.'),
    AIMessage('Jaipur is a beautiful city.'),
    HumanMessage('What should I eat today?'),
]

trimmed = trim_messages(
    history,
    max_tokens=40,                             # the budget
    token_counter=count_tokens_approximately,  # fast estimate; no API call
    strategy='last',                           # keep the most recent messages
    start_on='human',                          # never start mid-exchange with an AI reply
    include_system=True,                       # always keep the system prompt
)

print('before:', count_tokens_approximately(history), 'tokens,', len(history), 'messages')
print('after: ', count_tokens_approximately(trimmed), 'tokens,', len(trimmed), 'messages')
for m in trimmed:
    print(f'  {m.type:6} {m.content}')

before: 64 tokens, 6 messages
after:  22 tokens, 2 messages
  system You are a helpful assistant.
  human  What should I eat today?


In [3]:
MAX_TOKENS = 150  # small on purpose, so the effect shows up after a few turns


def chat_node(state: MessagesState):
    # Trim only what goes to the LLM. state['messages'] (and the checkpointer) keeps everything.
    trimmed = trim_messages(state['messages'], max_tokens=MAX_TOKENS,
                            token_counter=count_tokens_approximately,
                            strategy='last', start_on='human', include_system=True)
    return {'messages': [model.invoke(trimmed)]}


graph = StateGraph(MessagesState)
graph.add_node('chat_node', chat_node)
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer=InMemorySaver())
config = {'configurable': {'thread_id': 'trim-1'}}

In [4]:
def chat(text):
    result = chatbot.invoke({'messages': [HumanMessage(content=text)]}, config=config)
    reply = result['messages'][-1]
    print(f'You: {text}\nAI:  {reply.content[:120]}')
    print(f'     (sent {reply.usage_metadata["input_tokens"]} prompt tokens | stored {len(result["messages"])} messages)\n')


chat('Hi! My name is Priyansh.')
chat('What is my name?')

You: Hi! My name is Priyansh.
AI:  Hello Priyansh! 👋 How can I help you today?
     (sent 80 prompt tokens | stored 2 messages)



You: What is my name?
AI:  Your name is Priyansh.
     (sent 109 prompt tokens | stored 4 messages)



In [5]:
# A few more turns: stored messages keep growing, but the prompt stays under the budget
for city in ['Paris', 'Tokyo', 'Cairo', 'Lima']:
    chat(f'Give me one short fact about {city}.')

You: Give me one short fact about Paris.
AI:  Paris is home to the world’s oldest public library, the Bibliothèque nationale de France, founded in 1461.
     (sent 134 prompt tokens | stored 6 messages)



You: Give me one short fact about Tokyo.
AI:  Tokyo is the only city in the world that is both a capital city and the largest metropolitan area, with over 37 million 
     (sent 176 prompt tokens | stored 8 messages)



You: Give me one short fact about Cairo.
AI:  Cairo is built on the ancient city of Memphis, which was the capital of Egypt during the Old Kingdom and is one of the e
     (sent 193 prompt tokens | stored 10 messages)



You: Give me one short fact about Lima.
AI:  Lima is the only capital city in the world that is located entirely on a coastal plain, with its historic center perched
     (sent 181 prompt tokens | stored 12 messages)



In [6]:
# The trade-off: the name was in the oldest messages, which got trimmed away
chat('What is my name?')

You: What is my name?
AI:  I don’t have that information—so I’m not sure what your name is.
     (sent 186 prompt tokens | stored 14 messages)

